# Contact-center change analysis
Ask a plain-English question about a contact center and get a governed before/after number, the drivers behind it, and calls you can open. All data here is synthetic. Each section is one step; run the cells top to bottom.

## 0. Set up
A supervisor should be able to trust that every answer comes from the same governed metrics and the same call data, so the agent is built once and reused by every cell.

In [ ]:
import time
import uuid
import warnings

from langgraph.checkpoint.memory import InMemorySaver

from agent.citation_view import render_citation
from agent.graph import build_graph
from agent.tools import MCPToolbox
from evals.run import _ask, run_question
from rag.pipeline import get_llm
from retrieval import get_retriever

warnings.filterwarnings("ignore")
graph = build_graph(get_llm=get_llm, toolbox=MCPToolbox(), checkpointer=InMemorySaver())


def show(answer, limit=25):
    lines = (answer or "").splitlines()
    print("\n".join(lines[:limit]))
    if len(lines) > limit:
        print(f"... ({len(lines) - limit} more lines)")
    # the causation caveat closes every change answer, so keep it visible past the cap
    for line in lines[limit:]:
        if "the change caused" in line:
            print(line)


def headline(result):
    windows = (result.get("change") or {}).get("windows")
    if not windows:
        print("HEADLINE: no number (see the answer)")
        return
    before, after = windows["before"], windows["after"]
    top = (result["change"].get("drivers") or [{}])[0]
    print(f"HEADLINE: {before['value']:.2f} (n={before['n']}) -> {after['value']:.2f} (n={after['n']}), "
          f"change {after['value'] - before['value']:+.2f}"
          + (f"; top driver: {top.get('label')} ({top['after']['n']} calls)" if top.get("label") else ""))


def asks_first(result):
    for item in result.get("interrupts") or []:
        print(f"ASKS FIRST: \"{item['term']}\" matches {len(item['options'])} measures. Which do you mean?")
        for option in item["options"]:
            print("  -", option)
        print(f"(answering with {item['options'][0]})\n")


print("ready")

## 1. Why did a score fall?
Leaders need the size of a drop and what members were actually saying, not a hunch. The before/after comes from the governed CSAT metric, and each driver is counted as n of M calls with calls you can open.

In [ ]:
FEE_Q = "Why did satisfaction on fee calls drop after the fee schedule change?"
config = {"configurable": {"thread_id": str(uuid.uuid4())}}   # kept for the follow-up in step 2
t = time.perf_counter()
fee = await _ask(graph, FEE_Q, None, config)
print(f"question: {FEE_Q}\n")
headline(fee)
for side in ("before", "after"):
    print(f"\n{side} SQL:")
    show(fee["change"]["windows"][side]["sql"], 6)
print()
show(fee["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")

## 2. Open one cited call
Trust comes from checking the evidence. Any cited call can be opened with the quote highlighted, and account numbers are masked to their last four digits.

In [ ]:
cite = next(d["citations"][0] for d in fee["change"]["drivers"] if d.get("citations"))
hit = get_retriever().get_by_id(cite["call_id"])
print(render_citation(cite["call_id"], cite["quote"], hit.metadata, hit.text))

## 3. What should supervisors do about it?
A number is only useful if it leads to action. The follow-up reuses step 1's conversation and looks for patterns across calls, such as an explanation that changes from agent to agent.

In [ ]:
COACH_Q = "What should supervisors coach on for fee calls?"
t = time.perf_counter()
coach = await _ask(graph, COACH_Q, None, config)       # same thread as step 1
print(f"question: {COACH_Q}\n")
show(coach["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")

## 4. A change type it was never tuned on: online banking
A tool that only works on the example it was built for is a demo, not a product. Here the same agent takes a different kind of change and a different metric, call volume.

In [ ]:
Q = "Why did online banking calls go up in late July?"
t = time.perf_counter()
ob = await run_question(graph, Q)
print(f"question: {Q}\n")
headline(ob)
print()
show(ob["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")

## 5. Another one it was never tuned on: Westside members
Members of one branch often feel a change first. The question names no event, so the agent has to find the change point from the data itself.

In [ ]:
Q = "Why is satisfaction down for Westside members?"
t = time.perf_counter()
ws = await run_question(graph, Q)
print(f"question: {Q}\n")
headline(ws)
print()
show(ws["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")

## 6. Something that never happened
The costliest failure is a confident explanation for an event that did not occur. When the data shows no change point, the agent should stop and say so, with no number.

In [ ]:
Q = "Why did rate shopping calls go up after a competitor CD promotion?"
t = time.perf_counter()
cd = await run_question(graph, Q)
print(f"question: {Q}\n")
headline(cd)
print()
show(cd["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")

## 7. An ambiguous question
"Fees" could mean several different measures, and guessing wrong gives a confident but wrong number, so the agent asks which one first. Note too that every change answer above says the timing fits but the data cannot show that the change caused the shift.

In [ ]:
Q = "What do we make on fees?"
t = time.perf_counter()
amb = await run_question(graph, Q)
print(f"question: {Q}\n")
asks_first(amb)
show(amb["answer"])
print(f"\n[{time.perf_counter() - t:.0f}s]")